In [1]:
import re
import time
import html
import locale
import requests
from datetime import datetime
from bs4 import BeautifulSoup
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.keys import Keys 

from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

HEADERS = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.0.0 Safari/537.36",
             'cookie': 'over18=1;'}



In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
cd '/content/gdrive/My Drive/Colab Notebooks'

In [2]:
def webCrawl_104(pages):

    jobTitle = []
    companyName = []
    trade = []
    duty = []
    onlineDate = []
    location = []
    #experience = []
    #education = []
    salary = []
    companyStandard = []
    applyCount = []
    jobContent = []
    jobCriteria = []
    
    # selenium settings
    options=Options()
    options.chrome_executable_path='./chromedriver'
    driver=webdriver.Chrome(options=options)

    for p in range(1, pages+1):
    
        # 職缺列表頁的 URL
        #url = f'https://www.104.com.tw/jobs/search/?ro=1&isnew=7&keyword=數據分析師&expansionType=area%2Cspec%2Ccom%2Cjob%2Cwf%2Cwktm&order=17&asc=0&page={p}&mode=s&jobsource=2018indexpoc&langFlag=0&langStatus=0&recommendJob=1&hotJob=1'
        url = f'https://www.104.com.tw/jobs/search/?ro=1&jobcat=2007001018%2C2007001021%2C2007001022%2C2007001012%2C2007001020&expansionType=area%2Cspec%2Ccom%2Cjob%2Cwf%2Cwktm&order=16&asc=0&page={p}&mode=s&jobsource=2018indexpoc&langFlag=0&langStatus=0&recommendJob=1&hotJob=1'
        
        # 發送 HTTP 請求
        response = requests.get(url, headers=HEADERS)

        # 使用 BeautifulSoup 解析網頁原始碼
        soup = BeautifulSoup(response.text, 'html.parser')

        # 提取職缺標題、公司名稱、工作地點、薪資等資訊
        job_title = soup.find_all(class_='js-job-link')  # v
        company_name = soup.find_all('ul', class_='b-list-inline b-clearfix') # v
        
        date = soup.find_all('span', class_='b-tit__date') # 前三個空的 v
        date = date[3:]
        
        location_experience_education = soup.find_all(class_='b-list-inline b-clearfix job-list-intro b-content') # if-elif judge
        
        salary_companyStandard = soup.find_all('div', class_='job-list-tag b-content') # 前兩個空的 v
        salary_companyStandard = salary_companyStandard[2:]
        
        apply_count = soup.find_all('a', class_='b-link--gray gtm-list-apply') # v
        
        for n in range(len(job_title)):
            
            jobTitle.append(job_title[n].text)
            applyCount.append(apply_count[n].text)
            onlineDate.append(date[n].text.strip())
            
            # remember rewrite under loop
            #for n in company_name:
            company_name_lst = [i for i in company_name[n].text.split('\n') if i.strip()!='']
            
            companyName.append(company_name_lst[0])
            try:
                trade.append(company_name_lst[1])
            except:
                trade.append('-')
            
            #for n in range(len(location_experience_education)):
            location_experience_education_lst = [i for i in location_experience_education[n].text.split('\n')]

            location.append(location_experience_education_lst[1])
            #experience.append(location_experience_education_lst[3])
            #education.append(location_experience_education_lst[4])


            #for sc in salary_companyStandard:
            judge = ''.join(salary_companyStandard[n].text)
            #print(judge)
            lst = [i for i in salary_companyStandard[n].text.split(' ')]

            salary.append(lst[1])

            if '員工' in judge:

                index = [i for i, s in enumerate(lst) if '員工' in s]

                companyStandard.append(lst[index[0]])
                #test.append(lst[index[0]])

            else:
                companyStandard.append('-')
                
            # change url and get job content / criteria
            driver.get('https:'+job_title[n]['href'])

            # job content and requirement and duty
            try:
                jobs_duty = driver.find_element(By.CSS_SELECTOR, '.category-item.col.p-0.job-description-table__data')
                jd = jobs_duty.text.replace('認識','')
                jd = jd.replace('職務','')
                jd = jd.replace('\n','')
                duty.append(jd)
            except:
                duty.append('-')
            
            try:
                jobs_con = driver.find_element(By.CSS_SELECTOR, "div[class^='job-description-table']")
                jc = jobs_con.text.replace('\n','')
                jobContent.append(jc)
            except:
                jobContent.append('-')
                
            try:
                jobs_req = driver.find_element(By.CSS_SELECTOR, '.job-requirement-table.row')
                jr = jobs_req.text.replace('\n','')
                jobCriteria.append(jr)
                
            except:
                jobCriteria.append('-')

    driver.close()
    
    df = pd.DataFrame()
    
    df['職稱']=jobTitle
    df['工作內容']=jobContent
    df['條件需求']=jobCriteria
    df['公司名稱']=companyName
    df['產業']=trade
    df['職務']=duty
    df['上架日期']=onlineDate
    df['工作地']=location
    #df['經驗需求']=experience
    #df['學歷需求']=education
    df['薪資']=salary
    df['公司規模']=companyStandard
    df['應徵人數']=applyCount
    
    return df
                


In [ ]:
# input pages
test = webCrawl_104(2)
test

In [5]:
# 行銷專案類： CRM, 廣告投放, PM, GA, 社群 

profile_keyword_list = ['數據', '資料', 'AI','程式設計','Analytics','Data Analyst',
                        'Data Engineer','Data Science','商業分析','BI','市場分析','風險分析',
                        '金融分析','數據分析師','資料科學家','資料工程師','AI工程師','演算法工程師']

''' 104
數據分析師
資料科學家
資料工程師
AI工程師
演算法工程師
'''

''' 1111
數據科學家 v
數據分析師 v
數據管理師 
數據架構師
數據工程師 v
商業分析師
機器學習工程師
演算法開發工程師
統計分析師
數據建模工程師
臨床數據管理師
'''

# 判斷數據相關職缺
# def judge_data_related(df):
    
#     # 创建一个名为 'XX_Contains_Keyword' 的新列，检查 'Job_Title' 中是否包含关键字
#     df['JT_Contains_Keyword'] = df['職稱'].str.contains('|'.join(map(re.escape, profile_keyword_list)))
#     df['D_Contains_Keyword'] = df['職務'].str.contains('|'.join(map(re.escape, profile_keyword_list)))
    
#     # 使用布尔索引筛选包含关键字的行
#     filtered_df = df[df['JT_Contains_Keyword'] | df['D_Contains_Keyword']]
#     filtered_df = filtered_df.drop(columns=['JT_Contains_Keyword','D_Contains_Keyword'])

#     # 打印包含关键字的行
#     return filtered_df

def extract_information_jobContent(row):
    text = row['工作內容']
    
    try:
        jobContent = re.search(r'^(.*?)(職務類別|$)', text).group(1)
    except:
        jobContent = '不拘'
    try:
        duty = re.search(r'職務類別(.+?)工作待遇', text).group(1)
    except:
        duty = '不拘'
    try:
        salary = re.search(r'工作待遇(.+?)工作性質', text).group(1)
    except:
        salary = '不拘'
    try:
        jobType = re.search(r'工作性質(.+?)上班地點', text).group(1)
    except:
        jobType = '不拘'
    try:
        workCity = re.search(r'上班地點(.+?)管理責任', text).group(1)
    except:
        workCity = '不拘'
    try:
        manage = re.search(r'管理責任(.+?)出差外派', text).group(1)
    except:
        manage = '不拘'
    try:
        businessTrip = re.search(r'出差外派(.+?)上班時段', text).group(1)
    except:
        businessTrip = '不拘'
    try:
        workTime = re.search(r'上班時段(.+?)休假制度', text).group(1)
    except:
        workTime = '不拘'
    try:
        workDayOff = re.search(r'休假制度(.+?)可上班日', text).group(1)
    except:
        workDayOff = '不拘'
    try:
        workComein = re.search(r'可上班日(.+?)需求人數', text).group(1)
    except:
        workComein = '不拘'
    try:
        demandNum = re.search(r'需求人數(.+?)$', text).group(1)
    except:
        demandNum = '不拘'
    return pd.Series([jobContent, duty, salary, jobType, workCity, manage, businessTrip,
                     workTime, workDayOff, workComein, demandNum])

def extract_information_requirement(row):
    text = row['條件需求']
    
    try:
        experience = re.search(r'工作經歷(.+?)學歷要求', text).group(1)
    except:
        experience = '不拘'
    try:
        education = re.search(r'學歷要求(.+?)科系要求', text).group(1)
    except:
        education = '不拘'
    try:
        major = re.search(r'科系要求(.+?)語文條件', text).group(1)
    except:
        major = '不拘'
    try:
        language = re.search(r'語文條件(.+?)擅長工具', text).group(1)
    except:
        language = '不拘'
    try:
        tools = re.search(r'擅長工具(.+?)贊助', text).group(1)
    except:
        tools = '不拘'
    try:
        skills = re.search(r'工作技能(.+?)$', text).group(1)
    except:
        skills = '不拘'
    return pd.Series([experience, education, major, language, tools, skills])

# 应用提取函数并添加到 DataFrame 中
#test2 = judge_data_related(test)
test2 = test.copy()
test2[['工作經歷要求', '學歷要求', '科系要求', '語文條件', '擅長工具', '工作技能']] = test2.apply(extract_information_requirement, axis=1)
test2 = test2.drop(columns=['條件需求'])
test2[['工作敘述','職務類別','工作待遇','工作性質','上班地點','管理責任','出差外派','上班時段','休假制度','可上班日','需求人數']] = test2.apply(extract_information_jobContent, axis=1)
test2 = test2.drop(columns=['工作內容','職務類別'])
test2



,職稱,公司名稱,產業,職務,上架日期,工作地,薪資,公司規模,應徵人數,工作經歷要求,...,工作敘述,工作待遇,工作性質,上班地點,管理責任,出差外派,上班時段,休假制度,可上班日,需求人數
0,韌體工程師,凱納股份有限公司,-,韌體工程師、演算法工程師,,台中市南屯區,待遇面議,員工160人,0~5人應徵,3年以上,...,1. 單晶片程式設計: 進行STM32、Nordic系列晶片的韌體撰寫，為我們的電動自行車...,待遇面議 （經常性薪資達 4 萬元或以上）,全職,台中市南屯區精科南路99號 (精密機械科技創新園區),不需負擔管理責任,無需出差外派,日班,週休二日,不限,1人
1,數據工程師 Data Engineer,聚典資訊股份有限公司,電腦軟體服務業,資料工程師、資料科學家、數據分析師,10/03,台北市南港區,待遇面議,員工15人,11~30人應徵,不拘,...,【工作內容】1. 多數據源的數據清洗整理、串接客戶API數據、更新維護、自動化排程2. 資料...,待遇面議 （經常性薪資達 4 萬元或以上）,全職,台北市南港區園區街3-2號9樓912室 (南港軟體工業園區) (距捷運南港展覽館站約430公尺),不需負擔管理責任,無需出差外派,日班，9:00-18:00,週休二日,不限,1人
2,Data Scientist / Algorithm Engineer for Comput...,百威雷科技有限公司,其它軟體及網路相關業,演算法工程師、AI工程師、資料科學家,10/03,台北市中山區,"月薪70,000元以上",員工40人,11~30人應徵,不拘,...,[Job Title]: Data Scientist - Computer Vision[...,"月薪70,000元以上 （固定或變動薪資因個人資歷或績效而異）",全職,台北市中山區中山北路二段96號 (距捷運雙連站約270公尺),不需負擔管理責任,需出差，一年累積時間未定,日班，09:30~18:30,週休二日,不限,1人
3,資料工程師 ( Power BI/Tableau ),雲端達人科技股份有限公司,電腦軟體服務業,資料工程師、數據分析師,10/03,台北市中山區,待遇面議,員工70人,11~30人應徵,不拘,...,1. 資料維護與優化 - 內外部資料整合及收集2. 資料清理流程維護及優化 - 工作流程規劃...,待遇面議 （經常性薪資達 4 萬元或以上）,全職,台北市中山區林森北路577號11樓 (距捷運中山國小站約330公尺),不需負擔管理責任,無需出差外派,日班,依公司規定,不限,不限
4,數據分析師（歡迎遊戲PM轉職）,樂意傳播股份有限公司,數位內容產業,數據分析師、資料工程師、遊戲企劃,10/03,新北市新店區,待遇面議,員工180人,11~30人應徵,1年以上,...,1.負責對遊戲、營收、行銷等產品相關數據進行分析，並提供決策過程中可參考的分析報告。2.設計...,待遇面議 （經常性薪資達 4 萬元或以上）,全職,新北市新店區北新路三段205-3號9樓 (距捷運大坪林站約210公尺),不需負擔管理責任,無需出差外派,日班，09:00-18:00（另有彈性上下班）,週休二日,兩週內,1~2人
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2976,Quantitative Trading Summer Internship_暑期量化金融實...,Optiver_澳帝華期貨股份有限公司,其他金融及輔助業,金融交易員、金融研究員、演算法工程師,9/22,荷蘭,待遇面議,-,0~5人應徵,不拘,...,【申請說明】資格要求：大二以上在學生；申請時需上傳學校成績單(英文)及英文測驗成績單(如: ...,待遇面議 （經常性薪資達 4 萬元或以上）,全職,荷蘭,不需負擔管理責任,無需出差外派,日班,依公司規定,不限,1~5人
2977,統一企業-程式設計師/台南區永康區或台北市信義區,統一企業股份有限公司,飲料製造業,資料工程師、系統分析師、軟體工程師,9/22,台南市永康區,"月薪38,400~45,400元",員工5400人,0~5人應徵,不拘,...,資訊應用系統設計及程式設計與維護。工作地點:台南區永康區或台北市信義區※ 歡迎至統一企業人力...,"月薪38,400~45,400元 （固定或變動薪資因個人資歷或績效而異）",全職,台南市永康區或台北市信義區,不需負擔管理責任,需出差外派，一年累積時間未定,日班，08:00~17:00,依公司規定,一個月內,1人
2978,Index Options Trader_量化金融交易員,Optiver_澳帝華期貨股份有限公司,其他金融及輔助業,金融交易員、金融研究員、演算法工程師,9/22,台北市信義區,待遇面議,-,0~5人應徵,2年以上,...,"【申請說明】請勿於104投遞履歷, 煩請於底下網址申請https://optiver.com...",待遇面議 （經常性薪資達 4 萬元或以上）,全職,台北市信義區,不需負擔管理責任,無需出差外派,日班,依公司規定,不限,1~3人
2979,AI影像辨識應用工程師,智捷科技股份有限公司,通訊機械器材相關業,演算法工程師、軟體工程師,9/22,新竹市,待遇面議,員工100人,6~10人應徵,2年以上,...,1. 深度學習影像辨識相關應用(物件偵測、行為偵測)方案開發 2. 影像深度學習影像辨識演算...,待遇面議 （經常性薪資達 4 萬元或以上）,全職,新竹市新安路8號5樓 (新竹科學園區),不需負擔管理責任,無需出差外派,日班,週休二日,不限,1~2人


In [10]:
# 中英分離
demand = test2.filter(['職稱','職務','工作敘述'])

# 创建一个新的DataFrame来存放拆分后的行
new_rows = []

# 遍历原始DataFrame 並將 職務為 演算法工程師、軟體工程師 轉變成 兩筆資料
for index, row in demand.iterrows():
    job_titles = row['職務'].split('、')
    jobName = row['職稱']
    JD = row['工作敘述']
    for title in job_titles:
        new_row = {'職務': title, '職稱': jobName, '職缺描述':JD}
        new_rows.append(new_row)

# 创建包含新行的新DataFrame
new_demand = pd.DataFrame(new_rows)

data_duty = ['數據分析師','資料科學家','資料工程師','AI工程師','演算法工程師']
mask = new_demand['職務'].isin(data_duty)
new_demand = new_demand[mask]

# 將 工作敘述 的網址取代掉
def remove_url_and_html(text):
    # 用 re 取代 URL
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
    
    # 用 re 移除 HTML tag
    text = re.sub(r'<.*?>', '', text)
    text = text.replace('\n', '').replace('\r', '').replace('\t', '')
    
    return text

# use re to replace html entities
def replace_html_entities(text):
    
    text = re.sub(r'<.*?>', '', text)
    
    # parse entities
    text = html.unescape(text)
    
    return text

new_demand['職缺描述'] = new_demand['職缺描述'].apply(remove_url_and_html)
new_demand['職缺描述'] = new_demand['職缺描述'].apply(replace_html_entities)


# 中英文 分離
# 提取中文
def extract_chinese(text):
    chinese_pattern = r'[\u4e00-\u9fa5]+'  # 中文字符
    chinese_matches = re.findall(chinese_pattern, text)
    return ' '.join(chinese_matches)  # 將多個中文字符，以空格分隔

# 提取英文
def extract_english(text):
    english_pattern = r'[a-zA-Z]+'  # 英文字符
    english_matches = re.findall(english_pattern, text)
    return ' '.join(english_matches)  # 將多個英文字符，以空格分隔

new_demand['職缺描述（中文）'] = new_demand['職缺描述'].apply(extract_chinese)
new_demand['職缺描述（英文）'] = new_demand['職缺描述'].apply(extract_english)

new_demand


,職務,職稱,工作敘述,工作敘述（中文）,工作敘述（英文）
1,演算法工程師,韌體工程師,1. 單晶片程式設計: 進行STM32、Nordic系列晶片的韌體撰寫，為我們的電動自行車...,單晶片程式設計 進行 系列晶片的韌體撰寫 為我們的電動自行車提供先進的控制和功能 撰寫工程規...,STM Nordic
2,資料工程師,數據工程師 Data Engineer,【工作內容】1. 多數據源的數據清洗整理、串接客戶API數據、更新維護、自動化排程2. 資料...,工作內容 多數據源的數據清洗整理 串接客戶 數據 更新維護 自動化排程 資料庫 資料表設計且...,API airflow Linux Docker python pandas datetim...
3,資料科學家,數據工程師 Data Engineer,【工作內容】1. 多數據源的數據清洗整理、串接客戶API數據、更新維護、自動化排程2. 資料...,工作內容 多數據源的數據清洗整理 串接客戶 數據 更新維護 自動化排程 資料庫 資料表設計且...,API airflow Linux Docker python pandas datetim...
4,數據分析師,數據工程師 Data Engineer,【工作內容】1. 多數據源的數據清洗整理、串接客戶API數據、更新維護、自動化排程2. 資料...,工作內容 多數據源的數據清洗整理 串接客戶 數據 更新維護 自動化排程 資料庫 資料表設計且...,API airflow Linux Docker python pandas datetim...
5,演算法工程師,Data Scientist / Algorithm Engineer for Comput...,[Job Title]: Data Scientist - Computer Vision[...,,Job Title Data Scientist Computer Vision About...
...,...,...,...,...,...
7561,演算法工程師,Quantitative Trading Summer Internship_暑期量化金融實...,【申請說明】資格要求：大二以上在學生；申請時需上傳學校成績單(英文)及英文測驗成績單(如: ...,申請說明 資格要求 大二以上在學生 申請時需上傳學校成績單 英文 及英文測驗成績單 如 多益...,Quantitative Trading Summer Internship Ready t...
7562,資料工程師,統一企業-程式設計師/台南區永康區或台北市信義區,資訊應用系統設計及程式設計與維護。工作地點:台南區永康區或台北市信義區※ 歡迎至統一企業人力...,資訊應用系統設計及程式設計與維護 工作地點 台南區永康區或台北市信義區 歡迎至統一企業人力資...,
7567,演算法工程師,Index Options Trader_量化金融交易員,"【申請說明】請勿於104投遞履歷, 煩請於底下網址申請 Optiver, we seek t...",申請說明 請勿於 投遞履歷 煩請於底下網址申請,Optiver we seek to continuously improve our ap...
7568,演算法工程師,AI影像辨識應用工程師,1. 深度學習影像辨識相關應用(物件偵測、行為偵測)方案開發 2. 影像深度學習影像辨識演算...,深度學習影像辨識相關應用 物件偵測 行為偵測 方案開發 影像深度學習影像辨識演算法開發 影像...,python C C C java script AI


In [90]:
last_one = pd.read_excel('104數據職缺JD_中英文分離.xlsx')
last_one

,職務,職稱,工作敘述,工作敘述（中文）,工作敘述（英文）
0,數據分析師,C6251 商業智能工程師,遠傳電信個人家庭事業群正在尋找商業智能工程師(Business Intelligence E...,遠傳電信個人家庭事業群正在尋找商業智能工程師 你 妳將會成為數據分析團隊的一份子 負責打造各...,Business Intelligence Engineer user friendly S...
1,數據分析師,資料分析師,我們正在尋找一位對運動、健康、科技熱情的你加入Go-More團隊，能將滿滿數據變成一個會說的...,我們正在尋找一位對運動 健康 科技熱情的你加入 團隊 能將滿滿數據變成一個會說的故事 你渴望...,Go More PM App Web
2,資料科學家,資料分析師,我們正在尋找一位對運動、健康、科技熱情的你加入Go-More團隊，能將滿滿數據變成一個會說的...,我們正在尋找一位對運動 健康 科技熱情的你加入 團隊 能將滿滿數據變成一個會說的故事 你渴望...,Go More PM App Web
3,資料工程師,資料分析師,我們正在尋找一位對運動、健康、科技熱情的你加入Go-More團隊，能將滿滿數據變成一個會說的...,我們正在尋找一位對運動 健康 科技熱情的你加入 團隊 能將滿滿數據變成一個會說的故事 你渴望...,Go More PM App Web
4,AI工程師,AI實習生,我們歡迎大學在校生或即將畢業的學生申請，並且不限於特定科系。我們期望您具備對Python或C...,我們歡迎大學在校生或即將畢業的學生申請 並且不限於特定科系 我們期望您具備對 或 的程式語法...,Python C https ppt cc fPt lx
...,...,...,...,...,...
1558,演算法工程師,【Server】資料工程師,我們正在尋找對AI與虚擬人領域充滿熱情和專業知識的人員加入我們的團隊。作為AI資料工程師，您...,我們正在尋找對 與虚擬人領域充滿熱情和專業知識的人員加入我們的團隊 作為 資料工程師 您將負...,AI AI AI AI AI AI AI Python MySQL MongoDB
1559,演算法工程師,韌體工程師08,1. PCIE/SATA SSD 韌體開發2. USB 韌體開發3. Nand Flash ...,韌體開發 韌體開發 演算法開發,PCIE SATA SSD USB Nand Flash Sorting
1560,演算法工程師,直播串流研發工程師,浪LIVE AI團隊致力將最先進的人工智慧技術使用在直播場景上，過去一年我們自主研發了多項有...,浪 團隊致力將最先進的人工智慧技術使用在直播場景上 過去一年我們自主研發了多項有趣且富有挑戰...,LIVE AI AI AI C RTMP HLS WebRTC Kotlin Java Ob...
1561,演算法工程師,Data Scientist 資料科學家,1. Conduct data science projects to help clien...,NaN,Conduct data science projects to help clients ...


In [91]:
new_demand = pd.concat([last_one, new_demand])
new_demand

,職務,職稱,工作敘述,工作敘述（中文）,工作敘述（英文）
0,數據分析師,C6251 商業智能工程師,遠傳電信個人家庭事業群正在尋找商業智能工程師(Business Intelligence E...,遠傳電信個人家庭事業群正在尋找商業智能工程師 你 妳將會成為數據分析團隊的一份子 負責打造各...,Business Intelligence Engineer user friendly S...
1,數據分析師,資料分析師,我們正在尋找一位對運動、健康、科技熱情的你加入Go-More團隊，能將滿滿數據變成一個會說的...,我們正在尋找一位對運動 健康 科技熱情的你加入 團隊 能將滿滿數據變成一個會說的故事 你渴望...,Go More PM App Web
2,資料科學家,資料分析師,我們正在尋找一位對運動、健康、科技熱情的你加入Go-More團隊，能將滿滿數據變成一個會說的...,我們正在尋找一位對運動 健康 科技熱情的你加入 團隊 能將滿滿數據變成一個會說的故事 你渴望...,Go More PM App Web
3,資料工程師,資料分析師,我們正在尋找一位對運動、健康、科技熱情的你加入Go-More團隊，能將滿滿數據變成一個會說的...,我們正在尋找一位對運動 健康 科技熱情的你加入 團隊 能將滿滿數據變成一個會說的故事 你渴望...,Go More PM App Web
4,AI工程師,AI實習生,我們歡迎大學在校生或即將畢業的學生申請，並且不限於特定科系。我們期望您具備對Python或C...,我們歡迎大學在校生或即將畢業的學生申請 並且不限於特定科系 我們期望您具備對 或 的程式語法...,Python C https ppt cc fPt lx
...,...,...,...,...,...
2548,演算法工程師,AI技術專家 AI Technical Expert (桃園),1. AI演算法之研究、開發、設計、構建、驗證等工作。2. 跟蹤最前沿之AI深度學習算法技術...,演算法之研究 開發 設計 構建 驗證等工作 跟蹤最前沿之 深度學習算法技術並覆現 對需求模型...,AI AI Tensorflow Caffe PyTorch Python C
2551,演算法工程師,軟體開發工程師(實習、無薪)(新竹),物聯網實習生:對RPi 有興趣，做中學、學中做。Web 前端後端工程：以 Vue.js、C#...,物聯網實習生 對 有興趣 做中學 學中做 前端後端工程 以 創作需寫過作業 得錄取實習結束 ...,RPi Web Vue js C
2553,演算法工程師,前端系統區塊鏈應用工程師(Frontend) Blockchain Application ...,國泰自 2018 年成立區塊鏈團隊，從技術研究、底層開發、業務導入到跨鏈串接，一路走來持續推...,國泰自 年成立區塊鏈團隊 從技術研究 底層開發 業務導入到跨鏈串接 一路走來持續推出實際可落...,Web https recruit cathayholdings com CathaybkH...
2556,演算法工程師,【AR/MR】iPEBG SLAM演算法高級工程師,1.SLAM應用開發2.模型訓練調整3.軟體API設計開發4.與IMU進行系統整合、定位識別...,應用開發 模型訓練調整 軟體 設計開發 與 進行系統整合 定位識別與追蹤量測 開發 實施和優...,SLAM API IMU SLAM AR


In [11]:
new_demand.to_excel('104數據職缺JD_中英文分離_1003.xlsx',index=False)

In [66]:
# 計算職務間技能頻率
data_duty = ['數據分析師','資料科學家','資料工程師','AI工程師','演算法工程師']
correspond = {'擅長工具':'電腦技能','工作技能':'工作能力','專業證照':'證照'}

def duty_count_skill(df, col):
    
    duty = []
    skill = []
    
    for d in data_duty:
        df_d = df[df['職務'].str.contains(d)]
        df_d.reset_index(inplace = True)
        
        for i in range(df_d.shape[0]):
            skill_split = [s for s in df_d[col].iloc[i].split('、')]
            skill+=skill_split
            
            for t in range(len(skill_split)):
                duty.append(d)
            
    final_skill = pd.DataFrame({'職務':duty,
                                correspond[col]:skill})
    
    return final_skill

#arrange_df = duty_count_skill(test2,'工作技能')
arrange_df = duty_count_skill(test2,'擅長工具')
arrange_df
    

,職務,電腦技能
0,數據分析師,MS SQL
1,數據分析師,不拘
2,數據分析師,MySQL
3,數據分析師,Tableau
4,數據分析師,C#
...,...,...
1274,演算法工程師,不拘
1275,演算法工程師,不拘
1276,演算法工程師,Git
1277,演算法工程師,Python


In [67]:
#arrange_df = arrange_df[arrange_df['工作能力']!='不拘']
arrange_df = arrange_df[arrange_df['電腦技能']!='不拘']
arrange_df

,職務,電腦技能
0,數據分析師,MS SQL
2,數據分析師,MySQL
3,數據分析師,Tableau
4,數據分析師,C#
5,數據分析師,Java
...,...,...
1272,演算法工程師,Linux
1273,演算法工程師,Matlab
1276,演算法工程師,Git
1277,演算法工程師,Python


In [65]:
arrange_df.to_excel('104數據職缺_工作能力.xlsx')

In [19]:
test.to_excel('104_data_jobs.xlsx', index=False)

In [36]:
testtest = test2[test2['職務'].str.contains('數據分析師')]
testtest

,職稱,工作內容,公司名稱,產業,職務,上架日期,工作地,薪資,公司規模,應徵人數,工作經歷要求,學歷要求,科系要求,語文條件,擅長工具,工作技能
0,【開發金控】Data Analyst 數據分析人員,Data Analysis to support subsidiaries on busin...,中華開發金融控股股份有限公司(凱基證券)(凱基銀行)(中華開發資本),金融控股業,資料科學家、AI工程師、數據分析師,9/05,台北市南港區,待遇面議,員工21864人,0~5人應徵,5年以上,大學、碩士,不拘,英文 -- 聽 /中等、說 /中等、讀 /中等、寫 /中等贊助提升英文能力,不拘,不拘
1,IT數據分析_主任工程師,主要重點將是應用數據挖掘技術、進行統計分析以及構建與產品和製造流程整合的高品質預測系統。工作...,台橡股份有限公司,合成樹脂／塑膠及橡膠製造業,數據分析師、AI工程師、系統工程師,9/21,高雄市大社區,"月薪60,000元以上",員工700人,6~10人應徵,6年以上,大學以上,不拘,英文 -- 聽 /精通、說 /精通、讀 /精通、寫 /精通贊助提升英文能力,C#、Java、Python、SAP,不拘
4,數據分析師(有薪病假/生日假/特殊節日提早下班/小暑假/駐點按摩師),1. 熟悉排列組合、機率理論、可產生統計分析用的樣本資料、檢驗演算法與數據模型的正確性2. ...,唯數娛樂科技股份有限公司,電腦軟體服務業,數據分析師、資料科學家、資料工程師,9/20,台北市中正區,待遇面議,-,大於30人應徵,2年以上,大學以上,不拘,不拘,MySQL、Tableau,市場調查資料分析與報告撰寫、統計軟體操作、資料庫系統管理維護、報表彙整與管理
7,【個人金融】通路數據分析人員(經驗行員),1. 營業單位績效分析與模型建立2. 設計績效管理相關儀表板與數據串接3. 參與各式專案，並...,玉山銀行_玉山商業銀行股份有限公司,銀行業,數據分析師、統計學研究員、資料科學家,9/25,台北市松山區,待遇面議,員工8000人,0~5人應徵,不拘,大學、碩士,不拘,英文 -- 聽 /中等、說 /中等、讀 /中等、寫 /中等贊助提升英文能力,Python、Tableau,不拘
8,【海外】機械學習算法工程師,"1. 對數據敏感, 具有良好邏輯思維能力、業務理解能力及溝通表達能力2. 熟練使用Pytho...",健鼎科技股份有限公司,印刷電路板製造業(PCB),資料科學家、AI工程師、數據分析師,9/25,其他亞洲,待遇面議,員工25000人,6~10人應徵,2年以上,大學以上,數理統計相關、應用數學相關,不拘,Python,不拘
10,【PropTech數位人才招募】數據科學家/數據分析師,【關於PropTech數位人才招募計劃】信義企業集團的各項產品與計畫正融入大數據、機器學習與...,信義房屋股份有限公司,不動產經營業,資料科學家、演算法工程師、數據分析師,9/26,台北市信義區,待遇面議,員工5000人,11~30人應徵,3年以上,大學、碩士,數理統計相關、資訊管理相關、經濟社會及心理學科類,不拘,Python、R、Tableau、Power BI,不拘
13,【產品/研發類】產品專案工程師(自動化組),1.大數據資料分析及應用2.運用統計分析方法及機器學習提出適合演算模型3.自動控制與資料視覺...,欣興電子股份有限公司,印刷電路板製造業(PCB),自動控制工程師、數據分析師、資料工程師,9/26,桃園市龜山區,"月薪32,000~50,000元",員工15000人,0~5人應徵,不拘,大學、碩士,電機電子工程相關、資訊管理相關、資訊工程相關,英文 -- 聽 /中等、說 /中等、讀 /中等、寫 /中等贊助提升英文能力,Excel、Outlook、PowerPoint、Word,不拘
20,數據經營分析專員/課長,1. 集團營運數據分析及管理報表編制。2. 預算編制、財報分析。3. 建立數據庫，提供決策團...,山隆通運股份有限公司,汽車貨運業,主辦會計、財務分析／財務人員、數據分析師,9/21,新北市板橋區,待遇面議,員工1500人,6~10人應徵,1年以上,專科以上,不拘,不拘,Excel、Power BI,結帳作業與帳務處理、會計核算和帳務處理、編製帳務報表
21,專案專員,1.公司整體營運分析相關內容工作，包含：－營運相關數據維護工作。－定期（每週、每月）與工廠各...,勤美股份有限公司,其他金屬相關製造業,營運管理師／系統整合／ERP專案師、財務分析／財務人員、數據分析師,9/26,新竹縣新豐鄉,待遇面議,員工300人,11~30人應徵,5年以上,大學,不拘,英文 -- 聽 /略懂、說 /略懂、讀 /略懂、寫 /略懂台語 -- 中等贊助提升語文能力,Excel、PowerPoint、Tableau、Google Analytics、鼎新,不拘
24,研發-資料科學研究員,1. 執行AI技術開發專案。2. 大數據分析與建模。3. 舉辦內部統計與數值分析的教育訓練。...,達興材料股份有限公司,光電產業,統計學研究員、資料科學家、數據分析師,9/26,台中市西屯區,待遇面議,員工380人,11~30人應徵,不拘,碩士以上,資訊工程相關、數理統計相關、其他數學及電算機科學相關,英文 -- 聽 /精通、說 /精通、讀 /精通、寫 /精通贊助提升英文能力,Python、R,統計軟體操作、Machine Learning


In [ ]:
# 統一數據相關工作每個職務需要的唯一技能
data_category = {'數據分析師':['資料庫','Data Analyst']
                 '商業分析師':['BI','行銷數據','Business Analyst'],
                 '資料科學家':['演算法','Data Science','深度學習'],
                 '資料工程師':['數據工程','Data Engineer']}


In [ ]:
# 計算每個技能在不同職務之間被要求的次數



In [27]:
# dataframe = test.applymap(lambda x: x.encode('unicode_escape').
#                  decode('utf-8') if isinstance(x, str) else x)
# dataframe

In [38]:
date_max = test['上架日期'].max()
#date_min = test['上架日期'].min()
# current dateTime
now = datetime.now()
date_max_toDate = datetime.strptime(date_max, "%m/%d")
# convert to string
date_time_str = date_max_toDate.strftime("%m-%d")

import xlsxwriter
test.to_excel(f'104職缺_{date_time_str}_{str(current_dateTime.year)}.xlsx', index=False, engine='xlsxwriter')
